In [5]:
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [19]:
%load_ext autoreload
%autoreload 2
from drGAT import No_atten_drGAT
from drGAT.load_data import load_data
from drGAT.sampler import RandomSampler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
drugAct, pos_num, null_mask, S_d, S_c, S_g, A_cg, A_dg = load_data("ctrp")
PATH = "../ctrp_data/"

unique drugs: 10
unique genes: 17
DTI unique genes:  17
Top 90% variable genes:  1985
Total:  1999
Final gene exp shape: (823, 1999)
Final drug Act shape: (460, 823)


100%|██████████| 25/25 [00:02<00:00,  8.89it/s]


Done!


In [19]:
def objective(trial):
    params = {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "dropout1": trial.suggest_categorical("dropout1", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "dropout2": trial.suggest_categorical("dropout2", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "hidden1": trial.suggest_categorical(
            "hidden1",
            [256, 512, 1028],
        ),
        "hidden2": trial.suggest_categorical(
            "hidden2",
            [
                128,
                256,
                512,
            ],
        ),
        "hidden3": trial.suggest_categorical(
            "hidden3",
            [
                64,
                128,
                256,
            ],
        ),
        "epochs": trial.suggest_categorical("epochs", [10, 50, 100, 200, 500]),
        "heads": trial.suggest_categorical("heads", [1, 2, 3, 4, 5]),
        "activation": trial.suggest_categorical(
            "activation", ["relu", "gelu", "swish"]
        ),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical(
            "scheduler", [None, "Cosine", "Step", "Plateau"]
        ),
        "gnn_layer": trial.suggest_categorical(
            "gnn_layer",
            ["GCN", "MPNN"],
        ),
    }

    # スケジューラ関連パラメータの条件付き追加
    if params["scheduler"] == "Cosine":
        params["T_max"] = trial.suggest_int("T_max", 20, 50)
    elif params["scheduler"] == "Step":
        params["scheduler_gamma"] = trial.suggest_float("gamma_step", 0.1, 0.95)
        params["step_size"] = trial.suggest_int("step_size", 10, 30)
    elif params["scheduler"] == "Plateau":
        params["patience"] = trial.suggest_int("patience_plateau", 3, 10)
        params["threshold"] = trial.suggest_float(
            "thresh_plateau", 1e-4, 1e-2, log=True
        )

    if params["hidden1"] < params["hidden2"] or params["hidden2"] < params["hidden3"]:
        raise optuna.TrialPruned("Invalid layer size configuration")

    if params["optimizer"] in ["Adam", "AdamW"]:
        params["amsgrad"] = trial.suggest_categorical("amsgrad", [True, False])

    if params["optimizer"] == "SGD":
        params["momentum"] = trial.suggest_float("momentum", 0.8, 0.95)
        params["nesterov"] = trial.suggest_categorical("nesterov", [True, False])

    # 隠れ層サイズとバッチサイズの関係を制約
    if (params["hidden1"] > 512) and (params["hidden2"] > 256):
        raise optuna.TrialPruned("Memory intensive configuration")

    try:
        k = 5
        kfold = KFold(n_splits=k, shuffle=True, random_state=42)

        res = pd.DataFrame()
        for train_index, test_index in kfold.split(np.arange(pos_num)):
            sampler = RandomSampler(
                drugAct,
                train_index,
                test_index,
                null_mask,
                S_d,
                S_c,
                S_g,
                A_cg,
                A_dg,
                PATH,
            )
            _, best_metrics, _ = No_atten_drGAT.train(
                sampler, params=params, device=device, verbose=False
            )

            res = pd.concat(
                [
                    res,
                    pd.DataFrame(best_metrics, index=["acc", "f1", "auroc", "aupr"]).T,
                ]
            )

        return [float(i) for i in res.mean()]

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f"Pruned trial {trial.number}: CUDA OOM")

            # メモリ解放処理
            with torch.cuda.device("cuda"):
                torch.cuda.empty_cache()
            gc.collect()

            # Pruning通知
            raise optuna.TrialPruned(f"OOM at trial {trial.number}")

        else:
            raise e

In [20]:
name = "CTRP_GAT"
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage="sqlite:///{}.sqlite3".format(name),
    study_name=name,
    load_if_exists=True,
)
study.optimize(objective, n_trials=100)

[I 2025-03-21 17:01:55,450] Using an existing study with name 'CTRP_GAT' instead of creating a new one.
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/optuna/distributions.py:700: UserWarning: The distribution is specified by [10, 200] and step=50, but the range is not divisible by `step`. It will be replaced by [10, 160].
  warnings.warn(
[I 2025-03-21 17:01:55,908] Trial 2 pruned. Memory intensive configuration


Using:  cuda


100%|██████████| 60/60 [00:35<00:00,  1.69it/s]
[I 2025-03-21 17:02:32,191] Trial 3 finished with values: [0.829235960472403, 0.9014129679300726, 0.9025691346348286, 0.8286785152944022] and parameters: {'dropout1': 0.2, 'dropout2': 0.1, 'hidden1': 256, 'hidden2': 128, 'hidden3': 128, 'epochs': 60, 'heads': 2, 'activation': 'swish', 'optimizer': 'SGD', 'lr': 0.0016482007761046023, 'weight_decay': 0.0009291565621845429, 'scheduler': 'Step', 'gnn_layer': 'GATv2', 'gamma_step': 0.84042410266579, 'step_size': 10, 'momentum': 0.8002815301349936, 'nesterov': False, 'early_stop_threshold': 0.604442057849736}. 


Best model found at epoch 60
#####
[0.829235960472403, 0.9014129679300726, 0.9025691346348286, 0.8286785152944022]
#####


/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/optuna/distributions.py:700: UserWarning: The distribution is specified by [10, 200] and step=50, but the range is not divisible by `step`. It will be replaced by [10, 160].
  warnings.warn(
[I 2025-03-21 17:02:32,545] Trial 4 pruned. Invalid layer size configuration


...

CUDA out of memory


[W 2025-03-21 18:02:08,368] Trial 1036 failed with parameters: {'dropout1': 0.1, 'dropout2': 0.2, 'hidden1': 512, 'hidden2': 512, 'hidden3': 256, 'epochs': 110, 'heads': 3, 'activation': 'gelu', 'optimizer': 'SGD', 'lr': 0.0050452196599280255, 'weight_decay': 2.1954863063879936e-05, 'scheduler': None, 'gnn_layer': 'Transformer', 'momentum': 0.9077678735227763, 'nesterov': True} because of the following error: The value None could not be cast to float.
[W 2025-03-21 18:02:08,369] Trial 1036 failed with value [None, None, None, None].


Using:  cuda


  0%|          | 0/110 [00:00<?, ?it/s]
[W 2025-03-21 18:02:19,537] Trial 1037 failed with parameters: {'dropout1': 0.3, 'dropout2': 0.2, 'hidden1': 512, 'hidden2': 512, 'hidden3': 256, 'epochs': 110, 'heads': 3, 'activation': 'gelu', 'optimizer': 'SGD', 'lr': 0.007073005048561848, 'weight_decay': 0.0004397550346896202, 'scheduler': None, 'gnn_layer': 'Transformer', 'momentum': 0.9383363467029713, 'nesterov': True} because of the following error: The value None could not be cast to float.
[W 2025-03-21 18:02:19,537] Trial 1037 failed with value [None, None, None, None].


CUDA out of memory


## Eval

In [21]:
# test_drug = test_data.values[:, 0]
# test_cell = test_data.values[:, 1]

# test_labels = np.load("data/test_labels.npy")
# test_labels = torch.tensor(test_labels).float()
# test = [drug, cell, gene, edge_index, test_drug, test_cell, test_labels]

In [22]:
# prob, res, test_attention = drGAT.eval(model, test)